<a href="https://www.kaggle.com/code/asivakumarnair/diabetic-retinopathy-imagenet?scriptVersionId=344417511" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== STAGE 14 SESSION: FULL REBUILD + FINE-TUNE DEPTH ABLATION, RESNET50, POOLED DATA =====
# Three-way comparison: frozen (reload existing) / partial (new, conv4_block1 onward,
# contiguous-index unfreeze) / full (reload existing). Only "partial" needs new training,
# frozen and full are reloaded and evaluated with the same full metric set for a fair comparison.

!pip install -q tensorflow==2.19.0

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import random, gc
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")
assert 'tf_keras' in tf.keras.__name__, "STOP: Keras 3 active, not legacy. Restart before continuing."

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.model_selection import train_test_split
from sklearn.metrics import (cohen_kappa_score, roc_auc_score, accuracy_score,
                              f1_score, recall_score, confusion_matrix)

# ---------- CONFIG ----------
APTOS_CSV     = '/kaggle/input/competitions/aptos2019-blindness-detection/train.csv'
APTOS_IMG     = '/kaggle/input/competitions/aptos2019-blindness-detection/train_images'
EYEPACS_CSV   = '/kaggle/input/datasets/benjaminwarner/resized-2015-2019-blindness-detection-images/labels/trainLabels15.csv'
EYEPACS_IMG   = '/kaggle/input/datasets/benjaminwarner/resized-2015-2019-blindness-detection-images/resized train 15'
MESSIDOR_CSV  = '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor_data.csv'
MESSIDOR_IMG  = '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor-2/messidor-2/preprocess'
WEIGHTS_DIR   = '/kaggle/input/datasets/asivakumarnair/drbestmodels/'

GRADES, NUM_CLASSES = ['0','1','2','3','4'], 5
IMG_SIZE, BATCH_SIZE = 224, 32
SUBSAMPLE_SEED, EYEPACS_TARGET = 42, 3662
PHASE2_LR, EARLYSTOP_PAT, MONITOR = 1e-5, 7, 'val_accuracy'
UNFREEZE_FROM_BLOCK = 'conv4_block1'
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)

# ---------- POOLED DATA REBUILD ----------
aptos = pd.read_csv(APTOS_CSV)
aptos['grade']      = aptos['diagnosis'].astype(int).astype(str)
aptos['image_path'] = APTOS_IMG + '/' + aptos['id_code'].astype(str) + '.png'
aptos['patient_id'] = None

eyepacs = pd.read_csv(EYEPACS_CSV)
eyepacs['grade']      = eyepacs['level'].astype(int).astype(str)
eyepacs['image_path'] = EYEPACS_IMG + '/' + eyepacs['image'].astype(str) + '.jpg'
eyepacs['patient_id'] = eyepacs['image'].str.extract(r'^(\d+)_')

messidor = pd.read_csv(MESSIDOR_CSV)
messidor['grade']      = messidor['diagnosis'].astype(int).astype(str)
messidor['image_path'] = MESSIDOR_IMG + '/' + messidor['id_code'].astype(str)
messidor['patient_id'] = None

def subsample_eyepacs(df, target_n=EYEPACS_TARGET, seed=SUBSAMPLE_SEED):
    pg = df.groupby('patient_id')['grade'].max().reset_index()
    frac = target_n / len(df)
    keep, _ = train_test_split(pg, train_size=frac, stratify=pg['grade'], random_state=seed)
    return df[df['patient_id'].isin(keep['patient_id'])].reset_index(drop=True)

eyepacs_s = subsample_eyepacs(eyepacs)

def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError:
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_patient_level(df, rs=SEED, tag=""):
    pg = df.groupby('patient_id')['grade'].max().reset_index()
    p_tr, p_tmp = safe_split(pg, 'grade', 0.30, rs, tag=f"{tag} first")
    p_va, p_te  = safe_split(p_tmp, 'grade', 0.50, rs, tag=f"{tag} second")
    pick = lambda ids: df[df['patient_id'].isin(ids['patient_id'])]
    return pick(p_tr), pick(p_va), pick(p_te)

def split_image_level(df, rs=SEED, tag=""):
    tr, tmp = safe_split(df, 'grade', 0.30, rs, tag=f"{tag} first")
    va, te  = safe_split(tmp, 'grade', 0.50, rs, tag=f"{tag} second")
    return tr, va, te

def split_messidor_mixed(df, rs=SEED, tag="Messidor"):
    is_im = ~df['image_path'].str.contains(r'\d{8}_\d+_\d+_PP\.png$', regex=True)
    im_df = df[is_im].copy()
    im_df['im_num'] = im_df['image_path'].str.extract(r'IM(\d+)\.JPG$').astype(int)
    im_df = im_df.sort_values('im_num').reset_index(drop=True)
    im_df['patient_id'] = 'messidor_pair_' + (im_df.index // 2).astype(str)
    pg = im_df.groupby('patient_id')['grade'].max().reset_index()
    p_tr, p_tmp = safe_split(pg, 'grade', 0.30, rs, tag=f"{tag} IM first")
    p_va, p_te  = safe_split(p_tmp, 'grade', 0.50, rs, tag=f"{tag} IM second")
    pick = lambda ids: im_df[im_df['patient_id'].isin(ids['patient_id'])]
    im_tr, im_va, im_te = pick(p_tr), pick(p_va), pick(p_te)
    date_df = df[~is_im]
    d_tr, d_va, d_te = split_image_level(date_df, rs, tag=f"{tag} date-style")
    cat = lambda a, b: pd.concat([a.drop(columns=['im_num']), b], ignore_index=True)
    return cat(im_tr, d_tr), cat(im_va, d_va), cat(im_te, d_te)

a_tr, a_va, a_te = split_image_level(aptos, tag="APTOS")
e_tr, e_va, e_te = split_patient_level(eyepacs_s, tag="EyePACS")
m_tr, m_va, m_te = split_messidor_mixed(messidor)

train_df = pd.concat([a_tr, e_tr, m_tr], ignore_index=True)
val_df   = pd.concat([a_va, e_va, m_va], ignore_index=True)
test_df  = pd.concat([a_te, e_te, m_te], ignore_index=True)
print(f"Pooled Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")

def make_gens():
    train_idg = ImageDataGenerator(preprocessing_function=res_pre, **AUG)
    eval_idg  = ImageDataGenerator(preprocessing_function=res_pre)
    common = dict(x_col='image_path', y_col='grade', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=GRADES, color_mode='rgb')
    tr = train_idg.flow_from_dataframe(train_df, shuffle=True, seed=SEED, **common)
    va = eval_idg.flow_from_dataframe(val_df, shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(test_df, shuffle=False, **common)
    return tr, va, te

tr, va, te = make_gens()

cls = np.array(GRADES)
from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=cls, y=train_df['grade'])
CLASS_WEIGHT = {i: w for i, w in enumerate(cw)}

# ---------- FULL METRIC SET, same as Stage 8/12 ----------
def macro_specificity(y_true, y_pred, n_classes=NUM_CLASSES):
    cm = confusion_matrix(y_true, y_pred, labels=range(n_classes))
    total = cm.sum(); specs = []
    for i in range(n_classes):
        tp = cm[i,i]; fn = cm[i,:].sum()-tp; fp = cm[:,i].sum()-tp
        tn = total-tp-fn-fp
        specs.append(tn/(tn+fp) if (tn+fp) > 0 else np.nan)
    return np.nanmean(specs)

def evaluate_full(model, depth_label):
    y_prob = model.predict(te, verbose=0)
    y_true = np.asarray(te.classes)
    y_pred = y_prob.argmax(axis=1)
    qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    try:
        auc = roc_auc_score(np.eye(NUM_CLASSES)[y_true], y_prob, average='macro', multi_class='ovr')
    except ValueError:
        auc = np.nan
    row = dict(depth=depth_label, qwk=qwk, macro_auc=auc,
               accuracy=accuracy_score(y_true, y_pred),
               macro_f1=f1_score(y_true, y_pred, average='macro'),
               macro_sensitivity=recall_score(y_true, y_pred, average='macro'),
               macro_specificity=macro_specificity(y_true, y_pred))
    print(f"{depth_label:8} QWK={row['qwk']:.4f} macroAUC={row['macro_auc']:.4f} "
          f"Acc={row['accuracy']:.4f} F1={row['macro_f1']:.4f} "
          f"Sens={row['macro_sensitivity']:.4f} Spec={row['macro_specificity']:.4f}")
    return row

results = []

# ================================================================
# ARM 1: FROZEN, reload existing checkpoint, evaluate only, no training
# ================================================================
print("\n===== ARM 1: FROZEN (reload dr_phase1_resnet50.keras, inference only) =====")
frozen_model = load_model(WEIGHTS_DIR + 'dr_phase1_resnet50.keras')
results.append(evaluate_full(frozen_model, 'frozen'))
del frozen_model; gc.collect(); tf.keras.backend.clear_session()
tr, va, te = make_gens()  # generators reset after clear_session

# ================================================================
# ARM 2: PARTIAL, new training, starts from the frozen checkpoint
# ================================================================
print(f"\n===== ARM 2: PARTIAL (unfreeze {UNFREEZE_FROM_BLOCK} onward, contiguous index) =====")
model = load_model(WEIGHTS_DIR + 'dr_phase1_resnet50.keras')
base = model.layers[0]

# Contiguous-index unfreeze, NOT name-matching. A frozen BatchNorm sitting in the
# backward gradient path crashes (no deterministic GPU kernel for its backprop in
# inference mode), so everything from the cut point to the end must be trainable,
# no exceptions, regardless of individual layer names.
first_idx = next(i for i, l in enumerate(base.layers) if l.name.startswith(UNFREEZE_FROM_BLOCK))
base.trainable = True
for i, layer in enumerate(base.layers):
    layer.trainable = (i >= first_idx)

trainable_layers = [l.name for l in base.layers if l.trainable]
frozen_bn_in_path = [l.name for i, l in enumerate(base.layers)
                     if i >= first_idx and 'BatchNormalization' in l.__class__.__name__ and not l.trainable]
print(f"Unfreeze starts at index {first_idx} ({base.layers[first_idx].name})")
print(f"Trainable backbone layers: {len(trainable_layers)} of {len(base.layers)}")
print(f"Frozen BatchNorm layers in the gradient path (MUST be empty): {frozen_bn_in_path}")
assert len(frozen_bn_in_path) == 0, "A frozen BatchNorm sits in the backward path, this will crash. Do not proceed."

model.compile(Adam(PHASE2_LR), 'categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
model.fit(tr, validation_data=va, epochs=60, class_weight=CLASS_WEIGHT,
          callbacks=[EarlyStopping(monitor=MONITOR, patience=EARLYSTOP_PAT, restore_best_weights=True),
                     ModelCheckpoint('/kaggle/working/dr_stage14_partial_resnet50.keras',
                                      monitor=MONITOR, save_best_only=True),
                     CSVLogger('/kaggle/working/dr_stage14_partial_resnet50_log.csv', append=False)],
          verbose=1)

results.append(evaluate_full(model, 'partial'))
del model, base; gc.collect(); tf.keras.backend.clear_session()
tr, va, te = make_gens()

# ================================================================
# ARM 3: FULL, reload existing checkpoint, evaluate only, no training
# ================================================================
print("\n===== ARM 3: FULL (reload dr_best_resnet50.keras, inference only) =====")
full_model = load_model(WEIGHTS_DIR + 'dr_best_resnet50.keras')
results.append(evaluate_full(full_model, 'full'))
del full_model; gc.collect(); tf.keras.backend.clear_session()

# ================================================================
# SAVE + SUMMARY
# ================================================================
ablation_df = pd.DataFrame(results)
ablation_df.to_csv('/kaggle/working/dr_stage14_ablation.csv', index=False)
print("\n===== STAGE 14 ABLATION, ResNet50, pooled data =====")
print(ablation_df.round(4).to_string(index=False))
print("\nSaved: dr_stage14_ablation.csv, dr_stage14_partial_resnet50.keras, dr_stage14_partial_resnet50_log.csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 73.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.19.0 which is incompatible.
tf-keras 2.20.0 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.19.0 which is incompatible.


2026-08-23 16:50:47.562925: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787503847.585544      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787503847.593522      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787503847.612099      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787503847.612117      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787503847.612119      23 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
Pooled Train 6,344 | Val 1,361 | Test 1,363
Found 6344 validated image filenames belonging to 5 classes.
Found 1361 validated image filenames belonging to 5 classes.
Found 1363 validated image filenames belonging to 5 classes.

===== ARM 1: FROZEN (reload dr_phase1_resnet50.keras, inference only) =====


I0000 00:00:1787503879.450944      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787503879.457276      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1787503892.953298      76 cuda_dnn.cc:529] Loaded cuDNN version 91002


frozen   QWK=0.6667 macroAUC=0.8294 Acc=0.6354 F1=0.4232 Sens=0.4613 Spec=0.8898
Found 6344 validated image filenames belonging to 5 classes.
Found 1361 validated image filenames belonging to 5 classes.
Found 1363 validated image filenames belonging to 5 classes.

===== ARM 2: PARTIAL (unfreeze conv4_block1 onward, contiguous index) =====
Unfreeze starts at index 81 (conv4_block1_1_conv)
Trainable backbone layers: 94 of 175
Frozen BatchNorm layers in the gradient path (MUST be empty): []
Epoch 1/60


I0000 00:00:1787503998.821165      74 service.cc:152] XLA service 0x7ad57d38e8a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787503998.821211      74 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1787503998.821217      74 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1787503998.973528      74 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


199/199 [==============================] - 610s 3s/step - loss: 1.2467 - accuracy: 0.5966 - auc: 0.8510 - val_loss: 0.8866 - val_accuracy: 0.6767 - val_auc: 0.9021
Epoch 2/60
199/199 [==============================] - 436s 2s/step - loss: 1.1046 - accuracy: 0.6576 - auc: 0.8890 - val_loss: 0.8946 - val_accuracy: 0.6672 - val_auc: 0.8991
Epoch 3/60
199/199 [==============================] - 431s 2s/step - loss: 1.0401 - accuracy: 0.6652 - auc: 0.8966 - val_loss: 0.9166 - val_accuracy: 0.6481 - val_auc: 0.8927
Epoch 4/60
199/199 [==============================] - 430s 2s/step - loss: 0.9848 - accuracy: 0.6750 - auc: 0.9039 - val_loss: 0.8813 - val_accuracy: 0.6627 - val_auc: 0.8996
Epoch 5/60
199/199 [==============================] - 431s 2s/step - loss: 0.9524 - accuracy: 0.6690 - auc: 0.9042 - val_loss: 0.8740 - val_accuracy: 0.6620 - val_auc: 0.9011
Epoch 6/60
199/199 [==============================] - 435s 2s/step - loss: 0.8993 - accuracy: 0.6824 - auc: 0.9143 - val_loss: 0.8846 - 